In [ ]:
# Install dependencies if needed:
# !pip install -r requirements.txt

## Scratchpad 

1. Let them know v3.1 deprecated - only have v5 and it requires bearer token 
2. Look for alternatives? 
3. Design decision — Selective source ingestion

The REST Countries ingestion process requests only attributes required for country identification and currency resolution. Geographic and demographic attributes are intentionally excluded because they do not support the defined exchange-rate use case. The selected API response is preserved in its original structure before downstream transformations.



In [ ]:
# Final decision: Don't use this API as this requires
# API token. 

import requests
import os
import json
from pathlib import Path
from dotenv import load_dotenv

# find the .env from the notebook's working directory
env_path = next(
    (folder / ".env" for folder in (Path.cwd(), *Path.cwd().parents)
     if (folder / ".env").is_file()),
    None,
)
if env_path is not None:
    load_dotenv(env_path)

API_KEY = os.getenv("countries_api_key")

In [ ]:
headers = {"Authorization": f"Bearer {API_KEY}",
            "Accept": "application/json",}

params = {
    "response_fields": (
        "names.common,"
        "names.official,"
        "codes.alpha_2,"
        "codes.alpha_3,"
        "region,"
        "subregion,"
        "capitals,"
        "population,"
        "currencies"
    ),
    "limit": 100,
    "offset": 0
}

In [ ]:

URL = "https://api.restcountries.com/countries/v5"

FIELDS = [
    "names.common",
    "names.official",
    "codes.alpha_2",
    "codes.alpha_3",
    "region",
    "subregion",
    "capitals",
    "population",
    "currencies",
]

headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Accept": "application/json",
}

countries = []
offset = 0
limit = 100

with requests.Session() as session:
    while True:
        params = {
            "response_fields": ",".join(FIELDS),
            "limit": limit,
            "offset": offset,
        }

        response = session.get(
            URL,
            headers=headers,
            params=params,
            timeout=30,
        )
        response.raise_for_status()

        payload = response.json()
        data = payload["data"]

        records = data["objects"]
        meta = data["meta"]

        countries.extend(records)

        print(
            f"Fetched {len(records)} records "
            f"(total so far: {len(countries)})"
        )

        if not meta["more"]:
            break

        if not records:
            raise ValueError(
                "API indicates more data, but returned an empty page"
            )

        offset += len(records)

# Verify completeness
if len(countries) != meta["total"]:
    raise ValueError(
        f"Expected {meta['total']} countries, "
        f"received {len(countries)}"
    )

# Save the source records without flattening
with open("countries_raw.json", "w", encoding="utf-8") as f:
    json.dump(countries, f, indent=2, ensure_ascii=False)

print(f"\nSuccessfully fetched {len(countries)} countries")
print("Saved to countries_raw.json")

# Inspect Peru
peru = next(
    (
        country
        for country in countries
        if country.get("codes", {}).get("alpha_3") == "PER"
    ),
    None,
)

print("\nPeru record:")
print(json.dumps(peru, indent=2, ensure_ascii=False))


In [ ]:
# MEta has:

# total
# count
# limit
# offset
# more
# request_id
# duration

## Alternative? 

In [ ]:
import requests


countries_url = "https://countries.dev/countries"
countries_fields = ["name", "alpha2Code", "alpha3Code", "currencies"]
with requests.Session() as session:
    response = session.get(
            countries_url,
            params={"fields": ",".join(countries_fields)},
            timeout=30,
        )

In [ ]:
# response.json()

## Frankfurter

In [ ]:
from datetime import date, timedelta


In [ ]:

API_URL = "https://api.frankfurter.dev/v2/rates"
START_DATE = date(2026, 1, 1)
params = {"base": "USD"}

In [ ]:
response = requests.get(
        API_URL,
        params={"base": "USD", "date": "2026-11-03"},
        timeout=60,
    )

In [ ]:
response = requests.get(
        API_URL,
        params={"base": "USD", "date": "2026-11-03"},
        timeout=60,
    )

In [ ]:
response_json = response.json()

In [ ]:
len(response_json)

In [ ]:
response_json

# Looking through thr duckdb connection

In [ ]:
# !pip install duckdb

In [ ]:
from pathlib import Path
import duckdb

warehouse_path = Path("data/warehouse/branch.duckdb").resolve()

# Read-only access prevents accidental changes.
con = duckdb.connect(str(warehouse_path), read_only=True)

In [ ]:
display(con.sql("""
    SELECT table_schema, table_name, column_name, data_type
    FROM information_schema.columns
    ORDER BY table_name, ordinal_position
    limit 10
"""))

In [ ]:
display(con.sql("""
    SELECT table_schema, table_name
    FROM information_schema.columns
    group by 1,2
    ORDER BY table_name
    limit 10
"""))

In [ ]:
display(con.sql("""
select * from mart.country_exchange_rates
where source_country_code  = 'BTN' 
and destination_country_code = 'PAK'
order by rate_date desc
limit 10"""))

In [ ]:
display(con.sql("""
select * from core.countries 
where country_code = 'PAK'
limit 10"""))

In [ ]:
display(con.sql("""select * from core.country_currencies
where country_code in (SELECT country_code FROM core.country_currencies 
group by 1 having count(*) > 1 ) """))


In [ ]:
display(con.sql("""SELECT rate_date, source_currency, destination_currency, exchange_rate
FROM mart.country_exchange_rates
WHERE source_country_code = 'PER'
  AND destination_country_code = 'AUS'
  AND source_currency = 'PEN'
  AND destination_currency = 'AUD'
ORDER BY rate_date DESC
LIMIT 2;"""))

In [ ]:
display(con.sql("""SELECT rate_date, source_currency, destination_currency, exchange_rate
FROM mart.country_exchange_rates
WHERE source_country_code = 'PER'
  AND destination_country_code = 'AUS'
  AND source_currency = 'PEN'
  AND destination_currency = 'AUD'
  AND rate_date = CURRENT_DATE - 1;"""))

In [ ]:
con.close()